# Alpha Evaluator — Full Pipeline

**Two portfolio modes:**
1. **Simple Rank Portfolio** — fast, rank-based market-neutral (cell 4)
2. **Conviction Portfolio** — full GAM-style pipeline using pre-computed raw alphas from `raw_alphas.csv` (cells 5+)


In [ ]:
import os, sys
import pandas as pd
import numpy as np
from IPython.display import Markdown, display
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = "/Users/ritikraj/Utkarsh:QRT_comp/QRT_comp"
sys.path.append(os.path.join(BASE_DIR, "phase2_qrt_challenge"))

from scripts.alpha_evaluator import (
    evaluate_single_alpha, 
    build_conviction_portfolio,
    evaluate_conviction_portfolio,
    Alpha101
)

RAW_CSV_PATH = os.path.join(BASE_DIR, "stores", "raw_alphas.csv")
README_PATH = os.path.join(BASE_DIR, "phase2_qrt_challenge", "alphas_performance.md")


In [ ]:
# Load data
print("Loading data...")
returns = pd.read_parquet(os.path.join(BASE_DIR, "stores", "returns.parquet"))
universe = pd.read_parquet(os.path.join(BASE_DIR, "stores", "universe_5m.parquet"))
yf_data = pd.read_pickle(os.path.join(BASE_DIR, "top_5000_yf_data.pkl"))

adj_close = yf_data.xs('Close', level='Price', axis=1).loc[:, lambda df: ~df.columns.duplicated()]
volume = yf_data.xs('Volume', level='Price', axis=1).loc[:, lambda df: ~df.columns.duplicated()]

print(f"Data loaded — {len(returns)} days, {len(returns.columns)} tickers")

# Load pre-computed raw alphas from CSV
print(f"Loading raw alphas from {RAW_CSV_PATH}...")
raw_alphas_all = pd.read_csv(RAW_CSV_PATH, header=[0, 1], index_col=0, parse_dates=True)
available = raw_alphas_all.columns.get_level_values(0).unique().tolist()
print(f"Available pre-computed alphas: {available}")


---
## Mode 1: Compute & Save a New Alpha (Simple Rank Portfolio)
Use this to compute a new alpha that isn't in `raw_alphas.csv` yet.

In [ ]:
alpha_number = 1

evaluate_single_alpha(
    alpha_num=alpha_number,
    yf_data_df=yf_data,
    returns_df=returns,
    universe_df=universe,
    raw_csv_path=RAW_CSV_PATH,
    readme_path=README_PATH
)


---
## Mode 2: Conviction Portfolio (from pre-computed alphas)

Uses the **already saved** raw alpha from `raw_alphas.csv` — no recomputation needed.

Pipeline: ADV filter → Beta neutralization → Inverse-vol weighting → Z-score conviction + hysteresis → Position limits → Dollar-neutral L/S


In [ ]:
# ══════════════════════════════════════════════════════════
# CONFIGURE PARAMETERS HERE
# ══════════════════════════════════════════════════════════

ALPHA_NUMBER = 11          # Must already exist in raw_alphas.csv

ENTRY_THRESHOLD = 2.0      # Z-score to enter a new position
EXIT_THRESHOLD = 0.5       # Z-score to hold an existing position
MIN_ADV_USD = 5_000_000    # Minimum 60-day ADV in USD
TARGET_GMV = 10_000_000    # Target gross market value
VOL_WINDOW = 20            # Realized vol lookback (days)
BETA_WINDOW = 250          # Rolling beta lookback (days)


In [ ]:
# Load pre-computed alpha from CSV (no recomputation!)
alpha_name = f"alpha_{ALPHA_NUMBER:03d}"
assert alpha_name in available, f"{alpha_name} not found in raw_alphas.csv. Run Mode 1 first to compute it."

raw_alpha = raw_alphas_all[alpha_name]  # Extract the single alpha's Date x Ticker DataFrame
print(f"Loaded {alpha_name} from CSV — shape: {raw_alpha.shape}")


In [ ]:
# Build conviction portfolio
print(f"Building conviction portfolio for {alpha_name}...")
portfolio = build_conviction_portfolio(
    raw_alpha=raw_alpha,
    returns_df=returns,
    universe_df=universe,
    adj_close=adj_close,
    volume=volume,
    entry_threshold=ENTRY_THRESHOLD,
    exit_threshold=EXIT_THRESHOLD,
    min_adv_usd=MIN_ADV_USD,
    target_gmv=TARGET_GMV,
    vol_window=VOL_WINDOW,
    beta_window=BETA_WINDOW
)

active_days = (portfolio.abs().sum(axis=1) > 0).sum()
print(f"Portfolio built. Active trading days: {active_days}/{len(portfolio)}")


In [ ]:
# Evaluate YoY performance
print(f"\nEvaluating {alpha_name} conviction portfolio...\n")
metrics, overall = evaluate_conviction_portfolio(
    portfolio=portfolio,
    returns_df=returns,
    raw_alpha=raw_alpha,
    label=alpha_name
)

print("=" * 65)
print(f"  {alpha_name.upper()} — CONVICTION (z_entry={ENTRY_THRESHOLD}, z_exit={EXIT_THRESHOLD})")
print("=" * 65)
print(f"  Overall Net Sharpe:   {overall['Net Sharpe']}")
print(f"  Overall Gross Sharpe: {overall['Gross Sharpe']}")
print(f"  Overall Turnover:     {overall['Turnover']}")
print(f"  Overall Mean IC:      {overall['IC']}")
print("-" * 65)

df_yoy = pd.DataFrame(metrics)
if 'Year' in df_yoy.columns:
    df_yoy = df_yoy.set_index('Year')
display(df_yoy)


---
## View All Saved Performance Results

In [ ]:
if os.path.exists(README_PATH):
    with open(README_PATH, "r") as f:
        display(Markdown(f.read()))
else:
    print("No performance file found yet.")
